# EDA - Home Credit Default Risk (Phase 1)

Analysis of the assembled client-level feature mart and raw applications: class balance,
missing values, distributions of key features, multicollinearity, and anomalies.
Conclusions are in the last section and in `docs/eda.md`.

Before running: `make download` (data) and `make features` (mart).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

from pd_scoring.eda import (
    days_employed_anomaly,
    missingness,
    target_rate,
    top_correlations,
)


def _project_root() -> Path:
    here = Path.cwd()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").exists():
            return cand
    return here


ROOT = _project_root()
DATA = ROOT / "data"
mart = pl.read_parquet(DATA / "processed" / "mart.parquet")
app = pl.read_csv(DATA / "raw" / "application_train.csv", infer_schema_length=20000)
print("mart:", mart.shape, "| application_train:", app.shape)

## 1. Class balance (default rate)

In [ ]:
tr = target_rate(mart)
print("Labeled applications:", int(tr["n"]))
print("Defaults:", int(tr["positives"]), "=> rate {:.2%}".format(tr["rate"]))
plt.figure(figsize=(4, 3))
plt.bar(
    ["no default", "default"],
    [tr["n"] - tr["positives"], tr["positives"]],
    color=["#4c78a8", "#e45756"],
)
plt.title("Class balance")
plt.tight_layout()
plt.show()

## 2. Missing values (top-20 columns)

In [ ]:
miss = missingness(mart)
miss_top = miss.head(20)
with pl.Config(tbl_rows=20):
    print(miss_top)

## 3. Distributions of key features

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(mart["EXT_SOURCE_2"].drop_nulls().to_numpy(), bins=50, color="#4c78a8")
ax[0].set_title("EXT_SOURCE_2")
ax[1].hist((-app["DAYS_BIRTH"] / 365).to_numpy(), bins=50, color="#54a24b")
ax[1].set_title("Age, years")
ratio = mart["CREDIT_INCOME_RATIO"].drop_nulls().to_numpy()
ax[2].hist(ratio, bins=50, range=(0, 20), color="#f58518")
ax[2].set_title("CREDIT / INCOME")
plt.tight_layout()
plt.show()

## 4. Multicollinearity / relationship with the target

In [ ]:
corr = top_correlations(mart, k=15)
corr_df = pl.DataFrame(
    {
        "feature": [c for c, _ in corr],
        "corr_with_target": [round(v, 4) for _, v in corr],
    }
)
print(corr_df)

## 5. DAYS_EMPLOYED anomaly (365243)

In [ ]:
anom = days_employed_anomaly(app)
print(
    "DAYS_EMPLOYED == 365243:",
    int(anom["anomaly_count"]),
    "=> {:.2%} of applications".format(anom["anomaly_frac"]),
)
print("In the mart it is moved into the DAYS_EMPLOYED_ANOM flag, the value is cleaned to null.")

## 6. Save conclusions to docs/eda.md

In [ ]:
n_features = len([c for c in mart.columns if c not in ("SK_ID_CURR", "TARGET", "is_train")])
lines = [
    "# EDA - conclusions (Home Credit, Phase 1)",
    "",
    f"- Mart: {mart.height} clients x {mart.width} columns ({n_features} features).",
    f"- Class balance: defaults {int(tr['positives'])} of {int(tr['n'])} "
    f"({tr['rate']:.2%}) - strong imbalance (~1:11), account for it in training/metrics.",
    f"- DAYS_EMPLOYED==365243 anomaly: {int(anom['anomaly_count'])} "
    f"({anom['anomaly_frac']:.1%}) - placeholder for the unemployed; moved into a flag and cleaned to null.",
    "- EXT_SOURCE_1/2/3 - the strongest predictors (max |corr| with the target), but with missing values.",
    "",
    "## Top-15 missing values",
    "",
]
for row in missingness(mart).head(15).iter_rows(named=True):
    lines.append(f"- `{row['column']}`: {row['null_frac']:.1%}")
lines += ["", "## Top-15 correlations with TARGET", ""]
for c, v in corr:
    lines.append(f"- `{c}`: {v:+.3f}")
(ROOT / "docs").mkdir(exist_ok=True)
(ROOT / "docs" / "eda.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("written", ROOT / "docs" / "eda.md")

## Conclusions

- **Strong class imbalance** (~8% defaults): training needs class weights /
  PR-oriented metrics (PR-AUC, KS, Gini), not just accuracy.
- **EXT_SOURCE_1/2/3** - the most predictive features, but with noticeable missing values ->
  careful handling of missing values matters (in WOE - a separate bin).
- **`DAYS_EMPLOYED==365243` anomaly** - a technical placeholder (the unemployed): cleaned
  to null + a `DAYS_EMPLOYED_ANOM` flag, otherwise it would distort tenure and derived features.
- Monetary/age distributions are skewed (long tail) -> in Phase 2 fine for GBDT,
  for the scorecard WOE binning helps.
- Some bureau/card aggregates are strongly correlated with each other (multicollinearity) -
  for the logistic scorecard select by IV/VIF, for GBDT not critical.